In [2]:
import mlflow
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from sklearn import linear_model
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('SCD')


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1779782786199, experiment_id='0', last_update_time=1779786251550, lifecycle_stage='active', name='SCD', tags={}, trace_location=None, workspace='default'>

In [ ]:
#Import data and convert the variable types to the correct format for modeling
scd = pd.read_excel('full_data.xlsx')
columns_to_clean = ['ACR','Creatinine','Alb/mg/l']
for col in columns_to_clean:
    if col in scd.columns:
        scd[col] = scd[col].astype(str).str.replace('<', '').str.replace('>', '')
scd['Creatinine'] = pd.to_numeric(scd['Creatinine'],errors = 'coerce')
scd['Alb/mg/l'] = pd.to_numeric(scd['Alb/mg/l'],errors = 'coerce')
scd['ACR'] = pd.to_numeric(scd['ACR'],errors = 'coerce')
# Assign HMOX1 genotype based on allele values
scd['HMOX1 GENOTYPE'] = np.where(
    (scd['HMOX1 ALLELE 1'] < 25) & (scd['HMOX1 ALLELE 2'] < 25),
    'SS',
    np.where(
        (scd['HMOX1 ALLELE 1'] > 25) & (scd['HMOX1 ALLELE 2'] < 25),
        'LS',
        np.where(
            (scd['HMOX1 ALLELE 1'] < 25) & (scd['HMOX1 ALLELE 2'] > 25),
            'SL',
            np.where(
                (scd['HMOX1 ALLELE 1'] > 25) & (scd['HMOX1 ALLELE 2'] > 25),
                'LL',
                None
            )
        )
    )
)
scd['HMOX1 GENOTYPE'].value_counts()
#scd.info()





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 31 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   DNA NUMBER        266 non-null    object 
 1   APOL1 GENOTYPE    266 non-null    object 
 2   HMOX1 ALLELE 1    265 non-null    float64
 3   HMOX1 ALLELE 2    265 non-null    float64
 4   HMOX1 GENOTYPE    251 non-null    object 
 5   HMOX1_code        265 non-null    float64
 6   GENDER            266 non-null    object 
 7   AGE(Years)        266 non-null    int64  
 8   ETHNIC Group      266 non-null    object 
 9   BP                266 non-null    object 
 10  SBP               266 non-null    int64  
 11  DBP               266 non-null    int64  
 12  WEIGHT/Kg         266 non-null    float64
 13  HEIGHT/m          266 non-null    float64
 14  BMI (Kg/m2)       266 non-null    float64
 15  prot              266 non-null    object 
 16  GB/Nitr           266 non-null    object 
 1

In [ ]:
scd1 =pd.get_dummies(scd, columns=['GENDER','APOL1 GENOTYPE','HMOX1 GENOTYPE','prot','GB/Nitr'])
#scd1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 65 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   DNA NUMBER                         266 non-null    object 
 1   HMOX1 ALLELE 1                     265 non-null    float64
 2   HMOX1 ALLELE 2                     265 non-null    float64
 3   HMOX1_code                         265 non-null    float64
 4   AGE(Years)                         266 non-null    int64  
 5   ETHNIC Group                       266 non-null    object 
 6   BP                                 266 non-null    object 
 7   SBP                                266 non-null    int64  
 8   DBP                                266 non-null    int64  
 9   WEIGHT/Kg                          266 non-null    float64
 10  HEIGHT/m                           266 non-null    float64
 11  BMI (Kg/m2)                        266 non-null    float64

In [50]:
#Remove unneeded columns and columns with
#  clinical factors that are measured only on a few subjects
scd2= scd1.drop(columns=["BP","DNA NUMBER","ACR_code","ETHNIC Group","HMOX1_code","LDH","Fetal Hb","Bili totale","bili DIRECT","BILI INDIRECT"])
scd2 = scd2.dropna(axis=0, how='any')
X = scd2.drop(columns=["GFR"])
y = scd2['GFR']
X = X.dropna(axis=0, how='any')
#X.info()
X_train_df, X_test_df, Y_train_df, Y_test_df = train_test_split(X, y, test_size=0.2, random_state=42)
#X_train_df.info()
mlflow.sklearn.autolog()
model = linear_model.LinearRegression()
fitted_model = model.fit(X_train_df, Y_train_df)
print("Fitted model object:", fitted_model)
print("Coefficients:", fitted_model.coef_)
print("Intercept:", fitted_model.intercept_)
print("Train R^2:", fitted_model.score(X_train_df, Y_train_df))
print("Test R^2:", fitted_model.score(X_test_df, Y_test_df))

2026/05/28 11:18:05 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8601d2d5cce740a1a642542c944226b0', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/28 11:18:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/usr/local/python/3.12.1/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing V

Fitted model object: LinearRegression()
Coefficients: [-1.12851982e+00 -2.39167006e-01 -2.21117023e+00 -1.78875014e-01
  3.34299213e-01  4.57301218e-01 -7.33673113e+02 -9.72677633e-01
 -5.30164694e-02 -3.04757708e-02 -2.00196131e-02 -5.11565153e+00
 -7.58300414e-01 -1.46917392e+02  8.53904342e+00 -4.61145616e+00
  1.45661261e-13  4.61145616e+00 -5.46259279e+00  5.45588110e+01
 -1.12105619e+00  2.06202102e+00 -2.62343958e+01  8.52651283e-14
 -2.38027873e+01  1.28201928e+01 -8.91673290e+00 -4.54005477e-01
 -4.03852720e+00 -9.01099692e+00  1.59505670e+01  1.41555109e+01
  6.38981652e-01 -4.36564551e+01  3.97777851e+01 -1.38168655e+01
 -2.13162821e-14 -2.54207611e+01 -1.35187295e+01 -2.61109851e+01
 -2.10786179e+01  2.58287200e+01 -3.55271368e-15 -1.07900515e+01
  1.64404499e+00  1.36372609e+01 -7.64301392e-01 -2.03594308e+01
  1.35063487e+01  1.35650260e+01 -4.48823452e+00  8.83312402e+00
  3.00671343e+01  1.54494529e+01]
Intercept: 138.55042992488112
Train R^2: 0.6355465526447591
Test R^

In [ ]:
mlflow.start_run()
mlflow.set_tag('developer','Oluyomi')
mlflow.log_param('data-path', '908D4730.xlsx')